In [1]:
# Les importations
from datasets import load_dataset,Dataset, DatasetDict, concatenate_datasets


## Normalization

In [2]:

import pandas as pd
import unicodedata
import re

def normaliser_texte(texte):
    texte = unicodedata.normalize("NFC", texte)

    # Uniformiser les tirets
    texte = re.sub(r"[‐-‒–—―-−]", "-", texte)

    # Uniformiser les apostrophes
    texte = re.sub(r"[‘’‚ʼ‛]", "'", texte)

    # Uniformiser les guillemets
    texte = re.sub(r"[“”„‟]", '"', texte)

    return texte.strip()


def compter_doublons(file_fr, file_arb):

    # 1. Lire et normaliser les fichiers
    with open(file_fr, "r", encoding="utf-8") as f:
        phrases_fr = [
            normaliser_texte(ligne.rstrip("\r\n"))
            for ligne in f
        ]

    with open(file_arb, "r", encoding="utf-8") as f:
        phrases_arb = [
            normaliser_texte(ligne.rstrip("\r\n"))
            for ligne in f
        ]

    # 2. Vérifier l'alignement
    if len(phrases_fr) != len(phrases_arb):
        raise ValueError(
            f"Nombre de lignes différent : {len(phrases_fr)} en français "
            f"et {len(phrases_arb)} en arabe."
        )

    # 3. Construire le DataFrame initial
    df_initial = pd.DataFrame({
        "fr": phrases_fr,
        "arb": phrases_arb
    })

    # 4. Identifier toutes les lignes appartenant à une paire dupliquée
    masque_doublons = df_initial.duplicated(
        subset=["fr", "arb"],
        keep=False
    )
    doublons = df_initial[masque_doublons].copy()

    # Nombre d'occurrences excédentaires
    nombre_doublons = int(
        df_initial.duplicated(
            subset=["fr", "arb"],
            keep="first"
        ).sum()
    )

    # Nombre de paires distinctes répétées
    nombre_paires_dupliquees = int(
        df_initial.groupby(["fr", "arb"])
        .size()
        .gt(1)
        .sum()
    )

    # 5. Supprimer les paires répétées en gardant la première occurrence
    df = df_initial.drop_duplicates(
        subset=["fr", "arb"],
        keep="first"
    ).reset_index(drop=True)

    # 6. Réécrire les fichiers d'origine avec les données nettoyées
    with open(file_fr, "w", encoding="utf-8", newline="\n") as f:
        f.write("".join(phrase + "\n" for phrase in df["fr"]))

    with open(file_arb, "w", encoding="utf-8", newline="\n") as f:
        f.write("".join(phrase + "\n" for phrase in df["arb"]))

    # 7. Retourner les résultats
    return {
        "dataframe": df,
        "doublons": doublons,
        "nombre_doublons": nombre_doublons,
        "nombre_paires_dupliquees": nombre_paires_dupliquees
    }


## Statistique

In [1]:

import re
import unicodedata

def statistiques_textes(file_fr, file_arb):

    def lire_texte(chemin):
        with open(chemin, "r", encoding="utf-8") as f:
            texte = f.read()

        # Normalisation Unicode
        texte = unicodedata.normalize("NFC", texte)

        return texte

    # 1. Lire les deux fichiers
    texte_fr = lire_texte(file_fr)
    texte_arb = lire_texte(file_arb)

    # 2. Combiner les deux textes
    texte_combine = texte_fr + "\n" + texte_arb

    # 3. Tokeniser les mots (lettres Unicode, avec apostrophes internes)
    mots = re.findall(
        r"[^\W\d_]+(?:['’][^\W\d_]+)*",
        texte_combine,
        flags=re.UNICODE
    )

    # # Uniformiser la casse pour éviter de compter Bonjour et bonjour séparément
    # mots = [mot.casefold() for mot in mots]

    nombre_tokens = len(mots)
    nombre_mots_uniques = len(set(mots))

    # Considérer chaque ligne non vide comme un segment
    segments = [
        ligne.strip()
        for ligne in texte_combine.splitlines()
        if ligne.strip()
    ]

    # Calculer le nombre de mots par segment
    longueurs_segments = []

    for segment in segments:
        mots_segment = re.findall(
            r"[^\W\d_]+(?:['’][^\W\d_]+)*",
            segment,
            flags=re.UNICODE
        )
        longueurs_segments.append(len(mots_segment))

    # Calculer la longueur moyenne
    longueur_moyenne = (
        sum(longueurs_segments) / len(longueurs_segments)
        if longueurs_segments else 0
    )

    # 5. Calculer le Type-Token Ratio
    ttr = nombre_mots_uniques / nombre_tokens if nombre_tokens else 0

    return {
        "number_of_unique_words": nombre_mots_uniques,
        "total_number_of_tokens": nombre_tokens,
        "average_sentence_length": round(longueur_moyenne, 2),
        "type_token_ratio": round(ttr, 4)
    }


In [3]:
PATH = "../Data/full_dataset/"


In [4]:
resultat = compter_doublons(
    PATH + "french_not_bible.txt",
    PATH + "arabic_not_bible.txt"
)

df = resultat["dataframe"]
doublons = resultat["doublons"]

# print("DataFrame complet :")
# print(df.head())

print("Nombre de lignes en double :", resultat["nombre_doublons"])
print("Nombre de paires répétées :", resultat["nombre_paires_dupliquees"])

print("Paires dupliquées :")
print(doublons)

Nombre de lignes en double : 0
Nombre de paires répétées : 0
Paires dupliquées :
Empty DataFrame
Columns: [fr, arb]
Index: []


In [ ]:
## Statistiques
stas = statistiques_textes(
    PATH + "V_francais_3.txt",
    PATH + "french_not_bible.txt"
)

stas

{'number_of_unique_words': 26754,
 'total_number_of_tokens': 743340,
 'average_sentence_length': 19.44,
 'type_token_ratio': 0.036}

### Chargement du dataset

In [5]:
def dataset_loader(path) :
    with open(path, "r") as file:
        data = file.readlines()
        
    data_clean = [line for line in data if line.strip()]

    return data_clean


In [6]:
PATH = "../Data/full_dataset/"
data_arb_bible = dataset_loader(PATH + "arabe_tchad3.txt")
data_fr_bible = dataset_loader(PATH + "V_francais_3.txt")

## Ceux non biblique
data_arb_not_bible = dataset_loader(PATH + "arabic_not_bible.txt")
data_fr_not_bible = dataset_loader(PATH + "french_not_bible.txt")

In [7]:
# Création du dictionnaire
dataset_dict_bible = {"translation": [{"arb": a, "fr": f} for a, f in zip(data_arb_bible, data_fr_bible)]}
dataset_dict_not_bible = {"translation": [{"arb": a, "fr": f} for a, f in zip(data_arb_not_bible, data_fr_not_bible)]}


### Diviosion du dataset en train, test et validation

In [8]:
def equal_split(data_dict1: dict, data_dict2: dict | None = None) -> DatasetDict:
    def split_dataset(data_dict: dict) -> DatasetDict:
        dataset = Dataset.from_dict(data_dict)
        train_test = dataset.train_test_split(
            test_size=0.2,
            seed=42,
            shuffle=True,
        )
        val_test = train_test["test"].train_test_split(
            test_size=0.5,
            seed=42,
            shuffle=True,
        )

        return DatasetDict({
            "train": train_test["train"],
            "val": val_test["train"],
            "test": val_test["test"],
        })

    datasets = [split_dataset(data_dict1)]

    if data_dict2 is not None:
        datasets.append(split_dataset(data_dict2))

    return DatasetDict({
        split: concatenate_datasets([dataset[split] for dataset in datasets])
        if len(datasets) > 1
        else datasets[0][split]
        for split in ("train", "val", "test")
    })


In [9]:
train_test_val_dataset = equal_split(dataset_dict_bible, dataset_dict_not_bible)
train_test_val_dataset

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 30596
    })
    val: Dataset({
        features: ['translation'],
        num_rows: 3825
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 3826
    })
})

In [10]:
train_test_val_dataset.save_to_disk("../Data/train_test_split")

Saving the dataset (0/1 shards):   0%|          | 0/30596 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3825 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3826 [00:00<?, ? examples/s]